# LangChain, RAG, Prompt Engineering & Production Deployment

# LangChain & Database Concepts

## 1.1 Types of Chatbots

### A. QA (Question Answering) Chatbot
A QA chatbot answers questions using the LLM's built-in knowledge.

**Workflow:** `User → LLM → Answer`

**Example**
- User: *What is Python?*
- Bot: *Python is a high-level programming language.*

**Pros:** Simple to build, fast responses
**Cons:** Cannot access private/updated information, may hallucinate

### B. QA Conversational Chatbot
Remembers previous messages and maintains conversation context.

**Workflow:** `User ↔ Chat History ↔ LLM`

**Example**
- User: *My name is Ali.*
- *(later)* User: *What is my name?*
- Bot: *Your name is Ali.*

**Use cases:** Customer support, personal assistants, ChatGPT-style apps

### C. QA Retrieval Chatbot (RAG)
Retrieves relevant information from external documents before answering.

```text
User Question → Retriever → Relevant Documents → LLM → Answer
```

**Example:** User asks *"What is your refund policy?"* → Retriever searches company documents → LLM answers using retrieved content.

### D. QA Retrieval Conversational Chatbot
Combines conversation memory **with** document retrieval (RAG). Can remember previous messages, search external documents, and generate context-aware answers.

```text
User → Conversation Memory → Retriever → Relevant Context → LLM → Answer
```

**Example:** User uploads a restaurant menu, then later asks *"Which dessert did we discuss earlier?"* — the chatbot uses both chat history and the uploaded menu.

## 1.2 What is Semantic Search?

**Semantic Search** finds information based on the *meaning* of the query rather than exact keyword matches, using embeddings and vector similarity.

**Example**
- Document: *"Python is widely used in artificial intelligence."*
- Query: *"Which language is popular for AI?"*

Even though the wording differs, semantic search retrieves the correct document because the meanings are similar.

**Advantages:** Understands context · Handles synonyms · More accurate than keyword search

## 1.3 Tools in LangChain

Tools let an LLM interact with external systems — APIs, databases, calculators, search engines, or custom Python functions. Without tools, an LLM can only generate text; with tools, it can perform actions and retrieve real-time information.

**Examples:** Calculator · Web Search · Database Query · Weather API · Python Function · SQL Database

### Types of Tools
- **Built-in Tools** — provided by LangChain integrations (SQL Database Tool, Search Tools, Python REPL Tool)
- **Custom Tools** — created by the developer for specific tasks
- **External API Tools** — call external services (Weather API, Google Search, GitHub API, REST APIs)


In [ ]:
from langchain_core.tools import tool

@tool
def multiply(a: int, b: int):
    """Multiply two numbers."""
    return a * b


## 1.4 How Tool Calling Works & Binding Tools with an LLM

```text
User Question → LLM → Needs External Tool?
                          ├── No  → Answer
                          └── Yes → Call Tool → Tool Result → LLM → Final Answer
```

**Example:** User asks *"What is 25 × 40?"* — the LLM recognizes a calculation is needed, calls a calculator tool, receives the result (1000), and responds with the final answer.


In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool

@tool
def multiply(a: int, b: int):
    return a * b

llm = ChatOpenAI(model="gpt-4o-mini")

llm_with_tools = llm.bind_tools([multiply])
# Now the model can call the `multiply` tool whenever required.


## 1.5 What is an Agent?

An **Agent** is an intelligent system that decides *which* tool to use and *when* to use it, based on the user's request. Unlike a simple chain, an agent reasons and makes decisions dynamically.

```text
User Question → Agent → Reasoning → Select Tool → Execute Tool → Tool Output → Final Answer
```

**Example:** User asks *"What's the weather in Lahore today?"* — the agent decides a Weather API is needed, calls the weather tool, receives the temperature, and generates the response.


In [ ]:
from langchain.agents import create_tool_calling_agent

agent = create_tool_calling_agent(
    llm=llm,
    tools=[multiply],
    prompt=prompt
)
# The agent can now automatically choose and use the `multiply` tool when appropriate.


# Retrieval-Augmented Generation (RAG) & LangChain

## 2.1 What is RAG?

**RAG (Retrieval-Augmented Generation)** combines **information retrieval** with **Large Language Models (LLMs)**. Instead of relying only on the LLM's built-in knowledge, RAG first retrieves relevant information from external documents (PDFs, databases, websites, etc.) and then uses that information to generate an accurate answer.

```text
User Question → Document Collection → Text Chunking → Embeddings
     → Vector Store (Index) → Retriever → Relevant Context → LLM → Final Answer
```

### Components of RAG

**A. Index** — stores document embeddings inside a vector database. Organizes documents for fast, efficient similarity search. *Example: convert a PDF into embeddings and store them in Pinecone or FAISS.*

**B. Retrieval** — the retriever finds the most relevant document chunks based on the user's query, comparing vector similarity instead of keywords.

**C. Generation** — the retrieved context is sent to the LLM, which generates the answer *using* the retrieved information, making responses more accurate and grounded.

### Advantages of RAG
- Uses up-to-date information
- Reduces hallucinations
- Supports private/company documents
- Doesn't require retraining the model
- Produces more reliable answers

## 2.2 Vector Stores

A **Vector Store** is a database that stores **embeddings (vectors)** instead of plain text. When a query is received: (1) convert the query into an embedding, (2) compare it with stored vectors, (3) return the most similar documents.

| Feature | Pinecone | Qdrant | FAISS |
|---|---|---|---|
| Type | Cloud | Open-source DB | Local library |
| Storage | Cloud | Local/Cloud | Local |
| Scalability | Very High | High | Medium |
| Metadata Filtering | Yes | Yes | Limited |
| Best Use | Production | Self-hosted apps | Research & local projects |

- **Pinecone** — fully managed, highly scalable, fast similarity search, supports millions of vectors, easy API integration. Best for production/cloud deployment.
- **Qdrant** — open-source, high performance, metadata filtering, REST & gRPC APIs, self-hosted or cloud.
- **FAISS** — Meta's open-source library, stores vectors locally, extremely fast, no cloud required. Ideal for research and prototypes.

## 2.3 Context Engineering

**Context Engineering** is the process of providing the LLM with the **right information at the right time** so it can generate better responses, instead of giving it everything.

**A. Memory Management** — allows the chatbot to remember previous conversations.
> Without memory: *"I don't know."* → With memory: *"Your name is Ali."*

**B. Dynamic Context Injection** — the system retrieves only the most relevant information and inserts it into the prompt before sending it to the LLM.
> Question: *"What are today's restaurant offers?"* → Retriever injects: *"Today's Offer: Buy 1 Get 1 Free Pizza."*

**C. Structured Context** — information organized into a structured format (JSON, tables, key-value pairs) before being passed to the model, for consistent, accurate responses.


In [ ]:
{
  "restaurant": "Grand Dastarkhwan",
  "offer": "20% Discount",
  "timing": "10 AM - 11 PM"
}


## 2.4 Chains in LangChain

A **Chain** is a sequence of steps where the output of one step becomes the input of the next — combining prompts, models, retrievers, and parsers into a workflow.

```text
User Question → Prompt Template → LLM → Output Parser → Final Response
```


In [ ]:
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser

prompt = PromptTemplate.from_template(
    "Explain {topic} in simple words."
)

llm = ChatOpenAI()

chain = prompt | llm | StrOutputParser()

response = chain.invoke({"topic": "Machine Learning"})

print(response)


**Advantages:** Modular and reusable · Easy to read and maintain · Combines multiple AI components seamlessly

## 2.5 Build a Simple QA Chatbot

A basic question-answer chatbot using LangChain and OpenAI:

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# Initialize LLM
llm = ChatOpenAI(model="gpt-4o-mini")

# Create prompt template
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful AI assistant."),
    ("human", "{question}")
])

# Output parser
parser = StrOutputParser()

# Create chain
chain = prompt | llm | parser

# Chat loop
while True:
    question = input("You: ")

    if question.lower() == "exit":
        break

    answer = chain.invoke({"question": question})

    print("Bot:", answer)


# Prompt Engineering, Structured Output & ML Evaluation Metrics

## 3.1 What is Prompt Engineering?

**Prompt Engineering** is the practice of designing effective prompts to guide an AI model toward producing accurate and useful responses.

### A. Zero-shot Prompting
The model is given only the task, without any examples.
> `Translate "Hello" into French.` → Output: `Bonjour`

**Use case:** Simple tasks where the model already knows the pattern.

### B. Few-shot Prompting
The prompt includes a few examples before asking the actual question.
```text
English: Cat → French: Chat
English: Dog → French: Chien
English: House → French:
```
Output: `Maison`

**Use case:** Helps the model understand the desired format or pattern.

### C. Chain of Thought (CoT)
The model is encouraged to reason step-by-step before giving the final answer.
```text
John has 5 apples. He buys 3 more. He gives away 2.
Let's think step by step.
```
Output: `5 + 3 = 8, 8 - 2 = 6, Answer: 6`

**Use case:** Complex reasoning, math, logic, multi-step problems.

## 3.2 Static vs Dynamic Prompts (LangChain)

**Static Prompt** — fixed text, no variables, simple applications.


In [ ]:
prompt = "Explain Artificial Intelligence."

**Dynamic Prompt** — uses variables that change based on user input. Flexible, reusable, most commonly used in LangChain.

In [ ]:
template = "Explain {topic} in simple words."

topic = "Machine Learning"
# Output becomes: "Explain Machine Learning in simple words."


## 3.3 Prompt Templates in LangChain

A `PromptTemplate` lets you create reusable prompts with placeholders.

In [ ]:
from langchain_core.prompts import PromptTemplate

prompt = PromptTemplate.from_template(
    "Explain {topic} in simple words."
)

prompt.invoke({"topic": "CNN"})
# Output: "Explain CNN in simple words."


**Advantages:** Reusable · Cleaner code · Dynamic input support

## 3.4 LangChain Message Types

LangChain uses different message classes for conversations.

In [ ]:
from langchain_core.messages import HumanMessage, AIMessage

# HumanMessage — represents input from the user
HumanMessage(content="What is AI?")

# AIMessage — represents the model's response
AIMessage(content="AI stands for Artificial Intelligence.")

# ToolMessage — contains output returned by external tools
# e.g. Calculator Result: 25  -> sent back to the LLM


## 3.5 Structured Output in LangChain

**Structured Output** means the model returns data in a fixed format (JSON or Python objects) instead of plain text.

Instead of: *"The customer is Ali and age is 20."*

Return:
```json
{"name": "Ali", "age": 20}
```

**Advantages:** Easier to process · Reliable · API friendly

## 3.6 `with_structured_output()` & Pydantic Models

LangChain uses Pydantic models to define the expected output structure.


In [ ]:
from pydantic import BaseModel

class Student(BaseModel):
    name: str
    age: int

llm.with_structured_output(Student)

# Output:
# Student(name="Ali", age=20)


**Benefits:** Automatic validation · Type safety · Consistent responses

## 3.7 LangChain Output Parsers

Output parsers convert LLM responses into usable formats.

| | `StringOutputParser` | `JSONOutputParser` |
|---|---|---|
| Returns | Plain text | JSON |
| Style | Simple, human-readable | Structured, machine-readable |

## 3.8 Classification Evaluation Metrics

**Accuracy** — overall correctness: `Accuracy = (TP + TN) / Total Predictions`
> Example: 100 predictions, 95 correct → Accuracy = 95%

**Precision** — how many predicted positives are actually positive: `Precision = TP / (TP + FP)`
> Use case: Spam email detection

**Recall** — how many actual positives are found: `Recall = TP / (TP + FN)`
> Use case: Disease detection

**F1 Score** — harmonic mean of Precision and Recall: `F1 = 2 × Precision × Recall / (Precision + Recall)`
> Used when classes are imbalanced.

## 3.9 Regression Metrics

- **MAE (Mean Absolute Error):** `MAE = Average(|Actual − Predicted|)` — less sensitive to outliers
- **MSE (Mean Squared Error):** `MSE = Average((Actual − Predicted)²)` — penalizes large errors
- **RMSE (Root Mean Squared Error):** `RMSE = √MSE` — same unit as the target variable, easy to interpret

| Metric | Sensitive to Outliers |
|---|---|
| MAE | Low |
| MSE | High |
| RMSE | High |

## 3.10 Bias–Variance Tradeoff

The balance between underfitting and overfitting.

- **High Bias** — model too simple → underfitting → poor training performance
- **High Variance** — model too complex → overfitting → good training but poor testing performance

A good model maintains a balance between bias and variance.

## 3.11 Confusion Matrix

A table used to evaluate classification models:

| | Predicted Positive | Predicted Negative |
|---|---|---|
| **Actual Positive** | TP | FN |
| **Actual Negative** | FP | TN |

It is the basis for calculating Accuracy, Precision, Recall, and F1-Score.

## 3.12 Weights & Biases (W&B)

**W&B** is an experiment tracking and MLOps platform used to monitor ML experiments. It helps with:
- Tracking training metrics (loss, accuracy)
- Logging hyperparameters
- Visualizing graphs
- Comparing different experiments
- Collaborating with teams
- Saving model checkpoints

> Example: comparing two models trained with different learning rates to see which performs better.

## 3.13 Epochs, Batches & Iterations

- **Epoch** — one complete pass through the entire training dataset.
- **Batch** — a small subset of the dataset processed at one time.
- **Iteration** — one update of the model's weights after processing a single batch.

`Iterations = Number of Samples / Batch Size`

**Example:** Dataset = 1,000 samples, Batch size = 100, Epochs = 5
→ Iterations per epoch = 1,000 / 100 = **10**
→ Total iterations = 10 × 5 = **50**

## 3.14 CNN (Convolutional Neural Network) Concepts

### Convolution Operation
Sliding a small matrix (filter/kernel) over an input image to detect features like edges, corners, textures, and shapes. The filter multiplies its values with corresponding image pixels, sums the results into one value, and repeats across the image — producing a **feature map**.

### Filters / Kernels
A small matrix (e.g. 3×3 or 5×5) used to detect specific patterns. Different filters detect edges, corners, textures, shapes, or objects — CNNs learn the best filter values automatically during training.

### Stride & Padding

| Term | Purpose |
|---|---|
| **Stride** | Controls how far the filter moves each step and reduces output size |
| **Padding** | Preserves border information and controls output dimensions |

- Stride = 1 → filter moves one pixel at a time
- Stride = 2 → filter skips a pixel, reducing output size
- Padding adds extra pixels (usually zeros) around the image to prevent loss of border information

### Max Pooling vs Average Pooling

Pooling reduces feature-map size while keeping important information.

| Max Pooling | Average Pooling |
|---|---|
| Takes the maximum value | Takes the average value |
| Preserves the strongest features | Preserves overall/average information |
| Most commonly used | Less commonly used |

> Example window `[[2,5],[7,1]]` → Max Pooling = 7, Average Pooling = (2+5+7+1)/4 = 3.75

### Feature Maps
The output produced after applying a filter to an input image — each map highlights a specific detected feature (edges, lines, textures). Deeper CNN layers produce increasingly complex feature maps that recognize higher-level objects (faces, cars, animals).

### Backpropagation
The learning algorithm used to train neural networks:
1. Make predictions using current weights.
2. Calculate the error (predicted vs actual).
3. Propagate the error backward through the network.
4. Update weights using gradient descent.
5. Repeat until error is minimized.

**Purpose:** minimize prediction error and improve model accuracy.

#  Production AI Systems & Deployment

## 4.1 What is a Production-Level AI System?

A **production-level AI system** is an AI application that is reliable, scalable, secure, and accessible to real users in a live environment. Unlike a prototype, it must handle:

- Many users simultaneously
- Errors gracefully
- Secure authentication
- Fast response times
- Monitoring and logging
- Easy deployment and updates

### Production AI Architecture

```text
                Users
                  │
                  ▼
        Load Balancer / API Gateway
                  │
                  ▼
             FastAPI Backend
                  │
      ┌───────────┴───────────┐
      ▼                       ▼
 Authentication         Rate Limiter
      │                       │
      ▼                       ▼
      LangChain / LangGraph Workflow
                  │
      ┌───────────┴───────────┐
      ▼                       ▼
   Vector Store          PostgreSQL
(Pinecone/Qdrant/FAISS)     Database
                  │
                  ▼
              LLM (OpenAI/Ollama)
                  │
                  ▼
             JSON Response
```

## 4.2 Model Latency, Scalability & Cost

### A. Latency
Time taken for the model to return a response after receiving a request. Lower latency → better UX.

**Ways to reduce latency:** smaller models · caching frequent responses · optimized prompts · streaming responses · faster vector search

### B. Scalability
The system's ability to handle increasing numbers of users/requests (e.g. 10 users vs 10,000 users).

**Methods:** horizontal scaling (multiple servers) · load balancing · cloud auto-scaling · database optimization

### C. Cost Considerations
Production AI systems incur costs for LLM API usage, cloud servers, vector databases, storage, and bandwidth.

**Cost optimization:** cache responses · use smaller models for simple tasks · limit unnecessary API calls · retrieve only relevant documents · compress prompts

## 4.3 Introduction to Docker

**Docker** packages an application and all its dependencies into a **container**, ensuring it runs consistently across environments.

**Benefits:** works the same everywhere · easy deployment · isolated environment · portable

```text
Application → Dockerfile → Docker Image → Docker Container
```


In [ ]:
FROM python:3.11

WORKDIR /app

COPY . .

RUN pip install -r requirements.txt

EXPOSE 8000

CMD ["uvicorn", "main:app", "--host", "0.0.0.0", "--port", "8000"]


In [ ]:
from slowapi import Limiter
from slowapi.util import get_remote_address

limiter = Limiter(key_func=get_remote_address)

@app.get("/chat")
@limiter.limit("10/minute")
def chat():
    return {"message": "Hello"}


## 4.7 What is CORS?

**CORS (Cross-Origin Resource Sharing)** is a browser security mechanism that controls **which frontend domains are allowed to access your backend API**. Without proper CORS configuration, browsers block requests from different origins.

**Example**
- Frontend: `http://localhost:3000`
- Backend: `http://localhost:8000`

The frontend cannot call the backend unless CORS allows it.

## 4.8 CORS Middleware in FastAPI

FastAPI provides `CORSMiddleware` to configure allowed origins, methods, headers, and credentials.


In [ ]:
from fastapi.middleware.cors import CORSMiddleware

app.add_middleware(
    CORSMiddleware,
    allow_origins=[
        "http://localhost:3000"
    ],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)


| Option | Purpose |
|---|---|
| `allow_origins` | Allowed frontend domains |
| `allow_methods` | Allowed HTTP methods |
| `allow_headers` | Allowed request headers |
| `allow_credentials` | Allows cookies and authentication headers |


In [ ]:
FROM python:3.11

WORKDIR /app

COPY . .

RUN pip install -r requirements.txt

EXPOSE 8000

CMD ["uvicorn", "main:app", "--host", "0.0.0.0", "--port", "8000"]


In [ ]:
from slowapi import Limiter
from slowapi.util import get_remote_address

limiter = Limiter(key_func=get_remote_address)

@app.get("/ask")
@limiter.limit("5/minute")
def ask():
    return {"message": "Success"}

# If a client exceeds 5 requests per minute:
# HTTP 429 Too Many Requests
